# Week 7 Exercise – Llama Price Prediction Fine‑Tuned 

This notebook evaluates a fine‑tuned open‑source Llama model for product price prediction using Ed's public `pricer-data` dataset.
It focuses on loading the fine‑tuned model, running batched predictions, and analyzing errors (MAE/RMSE/MAPE).

In [ ]:
# imports
import os
import re
from dataclasses import dataclass
from typing import List

import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import matplotlib.pyplot as plt

In [ ]:
# constants
BASE_MODEL = "meta-llama/Meta-Llama-3.1-8B"
FINETUNED_MODEL = "ed-donner/pricer-2024-09-13_13.04.39" 
DATASET_NAME = "ed-donner/pricer-data"

QUANT_4_BIT = True
MAX_TEST_ITEMS = 500  

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE


In [ ]:
# load dataset from HuggingFace 
raw_data = load_dataset(DATASET_NAME)
train_data = raw_data["train"]
test_data = raw_data["test"]

print(f"Train size: {len(train_data)}  |  Test size: {len(test_data)}")

test = test_data.select(range(min(MAX_TEST_ITEMS, len(test_data))))
print(f"Using {len(test)} test items for evaluation")

In [ ]:
# dataclass for convenience
@dataclass
class Item:
    title: str
    description: str
    price: float


items: List[Item] = [
    Item(
        title=item["title"],
        description=item.get("description", ""),
        price=float(item["price"]),
    )
    for item in test
]

len(items), items[0]

In [ ]:
# load fine‑tuned LLaMA with 4‑bit quantization (QLoRA weights baked in)
quant_config = BitsAndBytesConfig(
    load_in_4bit=QUANT_4_BIT,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(FINETUNED_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    FINETUNED_MODEL,
    quantization_config=quant_config if QUANT_4_BIT else None,
    device_map="auto" if QUANT_4_BIT else None,
)

model.eval()
print("Model and tokenizer loaded.")

In [ ]:
# pricing prompt + helpers

def build_pricing_prompt(item: Item) -> str:
    return (
        "<|system|>\nYou are an expert price estimator for retail products. Analyze the product description carefully and predict the most likely new retail price in USD.\n"
        "<|user|>\n"
        f"{item.title}\n{item.description}\n"
        "<|assistant|>\n"
    )


def extract_price(text: str) -> float:
    text = (text or "").replace("$", "").replace(",", "")
    m = re.search(r"[-+]?\d*\.\d+|\d+", text)
    return float(m.group(0)) if m else 0.0

In [ ]:
@torch.no_grad()
def predict_price(item: Item, max_new_tokens: int = 16) -> float:
    prompt = build_pricing_prompt(item)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    prompt_decoded = tokenizer.decode(inputs["input_ids"][0], skip_special_tokens=True)
    continuation = decoded[len(prompt_decoded) :]
    return extract_price(continuation)

In [ ]:
def evaluate_model(items: List[Item], limit: int | None = None, title: str = "Opyom Week 7 LLaMA pricer"):
    use_items = items[:limit] if limit else items
    y_true, y_pred = [], []

    for i, it in enumerate(use_items):
        try:
            pred = predict_price(it)
        except Exception as e:
            print(f"Error on item {i}: {e}")
            pred = 0.0
        y_true.append(it.price)
        y_pred.append(pred)

    y_true_np = np.array(y_true, dtype=float)
    y_pred_np = np.array(y_pred, dtype=float)

    mae = float(np.mean(np.abs(y_pred_np - y_true_np)))
    rmse = float(np.sqrt(np.mean((y_pred_np - y_true_np) ** 2)))
    with np.errstate(divide="ignore", invalid="ignore"):
        mp_arr = np.where(y_true_np != 0, np.abs((y_pred_np - y_true_np) / y_true_np), np.nan)
    mape = float(np.nanmean(mp_arr)) * 100.0

    print(f"\n📈 {title}")
    print(f"MAE : {mae:.2f}")
    print(f"RMSE: {rmse:.2f}")
    print(f"MAPE: {mape:.2f}%")

    # scatter plot
    try:
        plt.figure(figsize=(6, 6))
        plt.scatter(y_true_np, y_pred_np, alpha=0.6)
        mx = max(y_true_np.max() if y_true_np.size else 0, y_pred_np.max() if y_pred_np.size else 0)
        plt.plot([0, mx], [0, mx], "r--", label="Ideal")
        plt.xlabel("Actual Price")
        plt.ylabel("Predicted Price")
        plt.title(title)
        plt.legend()
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f"Plotting error: {e}")

    # show worst 5 items by absolute error
    abs_err = np.abs(y_pred_np - y_true_np)
    worst_idx = np.argsort(-abs_err)[:5]
    print("\nWorst 5 predictions:")
    for idx in worst_idx:
        print(f"- {use_items[idx].title[:60]} | true={y_true_np[idx]:.2f}, pred={y_pred_np[idx]:.2f}, err={abs_err[idx]:.2f}")

    return {"mae": mae, "rmse": rmse, "mape": mape}


metrics = evaluate_model(items, limit=200)